In [1]:
import pandas as pd
from datetime import datetime
from pathlib import Path

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print(" Libraries imported successfully")

 Libraries imported successfully


In [2]:
# ==============================================================================
# STATE CODES
# ==============================================================================
STATE_CODES = {
    '50': ('AK', 'Alaska'), '01': ('AL', 'Alabama'), '54': ('AS', 'American Samoa'),
    '02': ('AZ', 'Arizona'), '03': ('AR', 'Arkansas'), '04': ('CA', 'California'),
    '52': ('CZ', 'Canal Zone'), '05': ('CO', 'Colorado'), '06': ('CT', 'Connecticut'),
    '08': ('DC', 'District of Columbia'), '07': ('DE', 'Delaware'), '09': ('FL', 'Florida'),
    '10': ('GA', 'Georgia'), '55': ('GM', 'Guam'), '51': ('HI', 'Hawaii'),
    '14': ('IA', 'Iowa'), '11': ('ID', 'Idaho'), '12': ('IL', 'Illinois'),
    '13': ('IN', 'Indiana'), '15': ('KS', 'Kansas'), '16': ('KY', 'Kentucky'),
    '17': ('LA', 'Louisiana'), '20': ('MA', 'Massachusetts'), '19': ('MD', 'Maryland'),
    '18': ('ME', 'Maine'), '21': ('MI', 'Michigan'), '22': ('MN', 'Minnesota'),
    '24': ('MO', 'Missouri'), '23': ('MS', 'Mississippi'), '25': ('MT', 'Montana'),
    '26': ('NB', 'Nebraska'), '32': ('NC', 'North Carolina'), '33': ('ND', 'North Dakota'),
    '28': ('NH', 'New Hampshire'), '29': ('NJ', 'New Jersey'), '30': ('NM', 'New Mexico'),
    '27': ('NV', 'Nevada'), '31': ('NY', 'New York'), '34': ('OH', 'Ohio'),
    '35': ('OK', 'Oklahoma'), '36': ('OR', 'Oregon'), '37': ('PA', 'Pennsylvania'),
    '53': ('PR', 'Puerto Rico'), '38': ('RI', 'Rhode Island'), '39': ('SC', 'South Carolina'),
    '40': ('SD', 'South Dakota'), '41': ('TN', 'Tennessee'), '42': ('TX', 'Texas'),
    '43': ('UT', 'Utah'), '62': ('VI', 'Virgin Islands'), '45': ('VA', 'Virginia'),
    '44': ('VT', 'Vermont'), '46': ('WA', 'Washington'), '48': ('WI', 'Wisconsin'),
    '47': ('WV', 'West Virginia'), '49': ('WY', 'Wyoming')
}

# ==============================================================================
# GEOGRAPHIC DIVISIONS & REGIONS
# ==============================================================================
COUNTRY_DIVISIONS = {
    '0': ('Possessions', ['54', '52', '55', '53', '62']),
    # Region 1 - NORTH EAST
    '1': ('New England', ['06', '18', '20', '28', '38', '44']),
    '2':( 'Middle Atlantic', ['29', '31', '37']),
    # Region 2 - NORTH CENTRAL
    '3': ('East North Central', ['12', '13', '21', '34', '48']),
    '4': ('West North Central', ['14', '15', '22', '24', '26', '33', '40']),
    # Region 3 - SOUTH
    '5': ('South Atlantic', ['07', '08', '09', '10', '19', '32']),
    '6': ('East South Central', ['01', '16', '23', '41']),
    '7': ('West South Central', ['03', '17', '35', '42']),
    # Region 4 - WEST
    '8': ('Mountain', ['02', '05', '11', '25', '27', '30', '43', '49']),
    '9': ('Pacific', ['50', '04', '51', '36', '46'])
}

COUNTRY_REGIONS = {
    '0': 'Possessions', '1': 'Northeast', '2': 'North Central', 
    '3': 'South', '4': 'West'
}

# ==============================================================================
# AGENCY TYPES & POPULATION GROUPS
# ==============================================================================
AGENCY_INDICATORS = {
    '0': 'Covered-By Another Agency', '1': 'City', '2': 'County',
    '3': 'University or College', '4': 'State Police',
    '5': 'Other Agencies (e.g., fire marshal, hospital, airport)',
    '6': 'State Agency Enforcement Unit', '7': 'Tribal / BIA Agencies',
    '8': 'Federal Agency'
}

POPULATION_GROUPS = {
    '0': 'Possessions', '1A': 'Cities 1,000,000+', '1B': 'Cities 500,000-999,999',
    '1C': 'Cities 250,000-499,999', '2': 'Cities 100,000-249,999',
    '3': 'Cities 50,000-99,999', '4': 'Cities 25,000-49,999',
    '5': 'Cities 10,000-24,999', '6': 'Cities 2,500-9,999', '7': 'Cities <2,500',
    '8A': 'Non-MSA Counties 100,000+', '8B': 'Non-MSA Counties 25,000-99,999',
    '8C': 'Non-MSA Counties 10,000-24,999', '8D': 'Non-MSA Counties <10,000',
    '8E': 'Non-MSA State Police', '9A': 'MSA Counties 100,000+',
    '9B': 'MSA Counties 25,000-99,999', '9C': 'MSA Counties 10,000-24,999',
    '9D': 'MSA Counties <10,000', '9E': 'MSA State Police'
}

print(f"Loaded {len(STATE_CODES)} state codes")
print(f"Loaded {len(AGENCY_INDICATORS)} agency types")
print(f"Loaded {len(POPULATION_GROUPS)} population groups")

Loaded 56 state codes
Loaded 9 agency types
Loaded 20 population groups


In [3]:
# ==============================================================================
# UCR OFFENSE CODES
# ==============================================================================
UCR_OFFENSE_CODES = {
    # Crimes Against Persons
    '09A': 'Murder/Non-negligent Manslaughter', '09B': 'Negligent Manslaughter',
    '13A': 'Aggravated Assault', '13B': 'Simple Assault', '13C': 'Intimidation',
    '11A': 'Rape', '11B': 'Sodomy', '11C': 'Sexual Assault With An Object',
    '11D': 'Fondling', '100': 'Kidnapping/Abduction',
    
    # Crimes Against Property
    '120': 'Robbery', '220': 'Burglary/Breaking and Entering',
    '23A': 'Pocket-picking', '23B': 'Purse-snatching', '23C': 'Shoplifting',
    '23D': 'Theft from Building', '23E': 'Theft from Coin-Operated Machine',
    '23F': 'Theft from Motor Vehicle', '23G': 'Theft of Motor Vehicle Parts',
    '23H': 'All Other Larceny', '240': 'Motor Vehicle Theft',
    '200': 'Arson', '290': 'Destruction/Damage/Vandalism of Property',
    
    # Crimes Against Society
    '250': 'Counterfeiting/Forgery', '26A': 'False Pretenses/Swindle',
    '26B': 'Credit Card/ATM Fraud', '26C': 'Impersonation',
    '270': 'Embezzlement', '280': 'Stolen Property Offenses',
    '35A': 'Drug/Narcotic Violations', '35B': 'Drug Equipment Violations',
    '520': 'Weapon Law Violations', '370': 'Pornography/Obscene Material',
    '210': 'Extortion/Blackmail'
}

print(f"Loaded {len(UCR_OFFENSE_CODES)} UCR offense codes")

Loaded 34 UCR offense codes


In [5]:
# ==============================================================================
# BIAS MOTIVATION CODES
# ==============================================================================
BIAS_MOTIVATION_CODES = {
    # Race/Ethnicity
    '11': 'Anti-White', '12': 'Anti-Black or African American',
    '13': 'Anti-American Indian/Alaska Native', '14': 'Anti-Asian',
    '15': 'Anti-Multiple Races, Group', '16': 'Anti-Native Hawaiian/Pacific Islander',
    '31': 'Anti-Arab', '32': 'Anti-Hispanic or Latino',
    '33': 'Anti-Other Race/Ethnicity/Ancestry',
    
    # Religion
    '21': 'Anti-Jewish', '22': 'Anti-Catholic', '23': 'Anti-Protestant',
    '24': 'Anti-Islamic (Muslim)', '25': 'Anti-Other Religion',
    '26': 'Anti-Multiple Religions, Group', '27': 'Anti-Atheism/Agnosticism',
    '28': 'Anti-Church of Jesus Christ (LDS)', '29': "Anti-Jehovah's Witness",
    '81': 'Anti-Eastern Orthodox', '82': 'Anti-Other Christian',
    '83': 'Anti-Buddhist', '84': 'Anti-Hindu', '85': 'Anti-Sikh',
    
    # Sexual Orientation
    '41': 'Anti-Gay (Male)', '42': 'Anti-Lesbian',
    '43': 'Anti-LGBTQ+ (Mixed Group)', '44': 'Anti-Heterosexual',
    '45': 'Anti-Bisexual',
    
    # Disability
    '51': 'Anti-Physical Disability', '52': 'Anti-Mental Disability',
    
    # Gender & Gender Identity
    '61': 'Anti-Male', '62': 'Anti-Female',
    '71': 'Anti-Transgender', '72': 'Anti-Gender Non-Conforming'
}

BIAS_CATEGORIES = {
    'Race/Ethnicity': ['11', '12', '13', '14', '15', '16', '31', '32', '33'],
    'Religion': ['21', '22', '23', '24', '25', '26', '27', '28', '29', 
                 '81', '82', '83', '84', '85'],
    'Sexual Orientation': ['41', '42', '43', '44', '45'],
    'Disability': ['51', '52'],
    'Gender': ['61', '62'],
    'Gender Identity': ['71', '72']
}

def get_bias_category(bias_code):
    """Return bias category for a given code."""
    for category, codes in BIAS_CATEGORIES.items():
        if bias_code in codes:
            return category
    return 'Unknown'

print(f"Loaded {len(BIAS_MOTIVATION_CODES)} bias motivation codes")
print(f"Loaded {len(BIAS_CATEGORIES)} bias categories")

Loaded 34 bias motivation codes
Loaded 6 bias categories


In [6]:
# ==============================================================================
# LOCATION CODES
# ==============================================================================
LOCATION_CODES = {
    '01': "Air/Bus/Train Terminal", '02': 'Bank/Savings and Loan',
    '03': 'Bar/Nightclub', '04': 'Church/Synagogue/Temple',
    '05': 'Commercial/Office Building', '06': 'Construction Site',
    '07': 'Convenience Store', '08': 'Department/Discount Store',
    '09': "Drug Store/Dr.'s Office/Hospital", '10': 'Field/Woods',
    '11': 'Government/Public Building', '12': 'Grocery/Supermarket',
    '13': 'Highway/Road/Alley', '14': 'Hotel/Motel/Etc.',
    '15': 'Jail/Prison', '16': 'Lake/Waterway', '17': 'Liquor Store',
    '18': 'Parking Lot/Garage', '19': 'Rental Storage Facility',
    '20': 'Residence/Home', '21': 'Restaurant', '22': 'School/College',
    '23': 'Service/Gas Station', '24': 'Specialty Store (TV, Fur, Etc.)',
    '25': 'Other/Unknown', '37': 'Abandoned/Condemned Structure',
    '38': 'Amusement Park', '39': 'Arena/Stadium/Fairgrounds/Coliseum',
    '40': 'ATM Separate from Bank', '41': 'Auto Dealership New/Used',
    '42': 'Camp/Campground', '44': 'Daycare Facility',
    '45': 'Dock/Wharf/Freight/Modal Terminal', '46': 'Farm Facility',
    '47': 'Gambling Facility/Casino', '48': 'Industrial Site',
    '49': 'Military Installation', '50': 'Park/Playground',
    '51': 'Rest Area', '52': 'School - College/University',
    '53': 'School - Elementary/Secondary', '54': 'Shelter - Mission/Homeless',
    '55': 'Shopping Mall', '56': 'Tribal Lands', '57': 'Community Center'
}

# ==============================================================================
# VICTIM & OFFENDER CHARACTERISTICS
# ==============================================================================
VICTIM_TYPES = {
    'I': 'Individual', 'B': 'Business', 'F': 'Financial Institution',
    'G': 'Government', 'R': 'Religious Organization', 'S': 'Society/Public',
    'O': 'Other', 'U': 'Unknown'
}

OFFENDER_RACE_CODES = {
    'W': 'White', 'B': 'Black or African American',
    'I': 'American Indian/Alaska Native', 'A': 'Asian',
    'M': 'Group of Multiple Races', 'P': 'Native Hawaiian/Other Pacific Islander',
    'U': 'Unknown'
}

OFFENDER_ETHNICITY_CODES = {
    'H': 'Hispanic or Latino', 'N': 'Not Hispanic or Latino', 'U': 'Unknown'
}

# ==============================================================================
# ADMINISTRATIVE CODES
# ==============================================================================
DATA_SOURCE_CODES = {
    'D': 'Data entry form', 'F': 'Floppy diskette/Internet e-mail',
    'N': 'NIBRS Incident'
}

QUARTER_CODES = {
    '1': 'Q1', '2': 'Q2',
    '3': 'Q3', '4': 'Q4'
}

print(f"Loaded {len(LOCATION_CODES)} location codes")
print(f"Loaded {len(VICTIM_TYPES)} victim types")
print(f"Loaded {len(OFFENDER_RACE_CODES)} offender race codes")

Loaded 45 location codes
Loaded 8 victim types
Loaded 7 offender race codes


In [25]:
# ==============================================================================
# PARSER FUNCTIONS
# ==============================================================================

def get_field(line: str, start: int, end: int) -> str:
    """Extract field from fixed-width line (1-based indexing)."""
    if len(line) < end:
        return line[start-1:].rstrip('\n').strip()
    return line[start-1:end].strip()

# Field specifications
bh_specs = [
    (1, 2, 'record_type'), (3, 4, 'state_code'), (5, 13, 'ori'),
    (26, 33, 'date_ori_added'), (34, 41, 'date_ori_went_nibrs'),
    (42, 71, 'city_name'), (72, 73, 'state_abbr'),
    (74, 75, 'population_group'), (76, 76, 'country_division'),
    (77, 77, 'country_region'), (78, 78, 'agency_indicator'),
    (79, 79, 'core_city'), (97, 97, 'nibrs_flag'),
    (98, 105, 'inactive_date'), (106, 114, 'current_population'),
    (115, 117, 'ucr_county_code'), (118, 120, 'msa_code'),
    (121, 129, 'last_population'), (226, 229, 'master_file_year'),
    (238, 267, 'agency_name'), (268, 282, 'fips_counties')
]

ir_specs = [
    (1, 2, 'record_type'), (3, 4, 'state_code'), (5, 13, 'ori'),
    (14, 25, 'incident_number'), (26, 33, 'incident_date'),
    (34, 34, 'data_source'), (35, 35, 'quarter'),
    (36, 38, 'num_victims'), (39, 40, 'num_offenders'),
    (41, 41, 'offender_race'), (42, 44, 'ucr_offense_code_1'),
    (45, 47, 'num_victims_1'), (48, 49, 'location_code_1'),
    (50, 51, 'bias_motivation_1'), (52, 59, 'victim_types_1'),
    (302, 304, 'adult_victims'), (305, 307, 'juvenile_victims'),
    (308, 309, 'adult_offenders'), (310, 311, 'juvenile_offenders'),
    (312, 312, 'offender_ethnicity')
]

print(" Parser functions defined")
print(f" BH record layout: {len(bh_specs)} fields")
print(f" IR record layout: {len(ir_specs)} fields")

 Parser functions defined
 BH record layout: 21 fields
 IR record layout: 20 fields


In [26]:
# ==============================================================================
# PARSE MULTIPLE MASTER FILES (2021-2024)
# ==============================================================================

# Define file paths for each year
file_patterns = [
    "data/2021_HC_NATIONAL_MASTER_FILE.txt",
    "data/2022_HC_NATIONAL_MASTER_FILE.txt",
    "data/2023_HC_NATIONAL_MASTER_FILE.txt",
    "data/2024_HC_NATIONAL_MASTER_FILE.txt"
]

bh_records = []
ir_records = []

for file_path in file_patterns:
    print(f"\n📁 Reading file: {file_path}")
    
    if not Path(file_path).exists():
        print(f"⚠️  File not found: {file_path} - Skipping")
        continue
    
    current_bh_index = None
    file_bh_count = 0
    file_ir_count = 0
    
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        for lineno, raw_line in enumerate(f, start=1):
            line = raw_line.rstrip("\n")
            
            if not line:
                continue
            
            record_type = line[:2]
            
            if record_type == "BH":
                bh_data = {name: get_field(line, start, end) 
                          for start, end, name in bh_specs}
                bh_data["_bh_index"] = len(bh_records)
                bh_data["_bh_line_no"] = lineno
                bh_data["_source_file"] = file_path
                bh_records.append(bh_data)
                current_bh_index = bh_data["_bh_index"]
                file_bh_count += 1
                
            elif record_type == "IR":
                ir_data = {name: get_field(line, start, end) 
                          for start, end, name in ir_specs}
                ir_data["bh_index"] = current_bh_index
                ir_data["_ir_line_no"] = lineno
                ir_data["_source_file"] = file_path
                ir_records.append(ir_data)
                file_ir_count += 1
    
    print(f"   ✅ Parsed {file_bh_count:,} BH records, {file_ir_count:,} IR records")

# Convert to DataFrames
df_bh = pd.DataFrame(bh_records)
df_ir = pd.DataFrame(ir_records)

print(f"\n{'='*80}")
print(f"📊 TOTAL ACROSS ALL YEARS:")
print(f"   BH (Agency) records: {len(df_bh):,}")
print(f"   IR (Incident) records: {len(df_ir):,}")
print(f"{'='*80}")


📁 Reading file: data/2021_HC_NATIONAL_MASTER_FILE.txt
   ✅ Parsed 26,136 BH records, 11,064 IR records

📁 Reading file: data/2022_HC_NATIONAL_MASTER_FILE.txt
   ✅ Parsed 26,136 BH records, 11,898 IR records

📁 Reading file: data/2023_HC_NATIONAL_MASTER_FILE.txt
   ✅ Parsed 26,136 BH records, 12,034 IR records

📁 Reading file: data/2024_HC_NATIONAL_MASTER_FILE.txt
   ✅ Parsed 26,136 BH records, 11,679 IR records

📊 TOTAL ACROSS ALL YEARS:
   BH (Agency) records: 104,544
   IR (Incident) records: 46,675


In [ ]:
# ==============================================================================
# CLEAN BATCH HEADER (BH) DATAFRAME
# ==============================================================================

# Set index
df_bh.set_index('_bh_index', inplace=True)

# Replace empty strings with NA
df_bh.replace('', pd.NA, inplace=True)

# Convert dates
date_cols = ['date_ori_added', 'date_ori_went_nibrs', 'inactive_date']
for col in date_cols:
    df_bh[col] = pd.to_datetime(df_bh[col], format='%Y%m%d', errors='coerce')

# Convert population to numeric
for col in ['current_population', 'last_population']:
    df_bh[col] = pd.to_numeric(df_bh[col], errors='coerce')

print(" BH DataFrame cleaned")
print(f"   - Index: _bh_index")
print(f"   - Dates converted: {len(date_cols)} columns")
print(f"   - Numeric conversions: 2 population columns")

 BH DataFrame cleaned
   - Index: _bh_index
   - Dates converted: 3 columns
   - Numeric conversions: 2 population columns


In [28]:
# ==============================================================================
# CLEAN INCIDENT REPORT (IR) DATAFRAME
# ==============================================================================

# Convert incident date
df_ir['incident_date'] = pd.to_datetime(df_ir['incident_date'], 
                                        format='%Y%m%d', errors='coerce')

# Convert numeric fields
numeric_cols = ['num_victims', 'num_offenders', 'num_victims_1',
                'adult_victims', 'juvenile_victims', 
                'adult_offenders', 'juvenile_offenders']

for col in numeric_cols:
    if col in df_ir.columns:
        df_ir[col] = pd.to_numeric(df_ir[col], errors='coerce')

print(" IR DataFrame cleaned")
print(f"   - Date converted: incident_date")
print(f"   - Numeric conversions: {len(numeric_cols)} columns")
print(f"   - Date range: {df_ir['incident_date'].min()} to {df_ir['incident_date'].max()}")

 IR DataFrame cleaned
   - Date converted: incident_date
   - Numeric conversions: 7 columns
   - Date range: 2021-01-01 00:00:00 to 2024-12-31 00:00:00


In [29]:
# ==============================================================================
# DECODE BATCH HEADER (BH) FIELDS
# ==============================================================================

def decode_bh(df):
    """Decode all coded fields in Batch Header DataFrame."""
    df = df.copy()
    
    # Decode state
    df['state_name'] = df['state_code'].map(
        lambda x: STATE_CODES.get(x, ('Unknown', 'Unknown'))[1]
    )
    
    # Decode geography
    df['region_name'] = df['country_region'].map(COUNTRY_REGIONS)
    df['division_name'] = df['country_division'].map(
        lambda x: COUNTRY_DIVISIONS.get(x, ('Unknown', []))[0]
    )
    
    # Decode agency
    df['agency_type'] = df['agency_indicator'].map(AGENCY_INDICATORS)
    df['population_group_desc'] = df['population_group'].map(POPULATION_GROUPS)
    
    # Flags
    df['is_core_city'] = df['core_city'].map({'Y': True, 'N': False})
    df['is_nibrs_active'] = df['nibrs_flag'].map({'A': True, ' ': False, '': False})
    
    return df

# Apply decoding
df_bh_decoded = decode_bh(df_bh)

print(" BH DataFrame decoded")
print(f"   - Added state_name, region_name, division_name")
print(f"   - Added agency_type, population_group_desc")
print(f"   - Added flags: is_core_city, is_nibrs_active")

 BH DataFrame decoded
   - Added state_name, region_name, division_name
   - Added agency_type, population_group_desc
   - Added flags: is_core_city, is_nibrs_active


In [30]:
# ==============================================================================
# DECODE INCIDENT REPORT (IR) FIELDS
# ==============================================================================

def decode_offense_flexible(code):
    """Decode UCR offense code with flexible pattern matching."""
    if pd.isna(code) or code == '':
        return None
    if code in UCR_OFFENSE_CODES:
        return UCR_OFFENSE_CODES[code]
    elif str(code).startswith('23') and len(str(code)) == 3:
        return 'All Other Larceny (23H)'
    else:
        return None

def decode_ir(df):
    """Decode all coded fields in Incident Report DataFrame."""
    df = df.copy()
    
    # Decode state
    df['state_name'] = df['state_code'].map(
        lambda x: STATE_CODES.get(x, ('Unknown', 'Unknown'))[1]
    )
    
    # Decode administrative fields
    df['data_source_desc'] = df['data_source'].map(DATA_SOURCE_CODES)
    df['quarter_desc'] = df['quarter'].map(QUARTER_CODES)
    
    # Decode offense (with flexible pattern matching)
    df['offense_1_desc'] = df['ucr_offense_code_1'].apply(decode_offense_flexible)
    
    # Decode location and bias
    df['location_1_desc'] = df['location_code_1'].map(LOCATION_CODES)
    df['bias_1_desc'] = df['bias_motivation_1'].map(BIAS_MOTIVATION_CODES)
    df['bias_1_category'] = df['bias_motivation_1'].apply(get_bias_category)
    
    # Decode offender characteristics
    df['offender_race_desc'] = df['offender_race'].map(OFFENDER_RACE_CODES)
    df['offender_ethnicity_desc'] = df['offender_ethnicity'].map(OFFENDER_ETHNICITY_CODES)
    
    # Decode victim types
    df['victim_types_1_decoded'] = df['victim_types_1'].apply(
        lambda x: ', '.join([VICTIM_TYPES.get(c, 'Unknown') 
                            for c in str(x) if c.strip()])
    )
    
    # Calculate totals
    df['total_victims'] = df['adult_victims'].fillna(0) + df['juvenile_victims'].fillna(0)
    df['total_offenders'] = df['adult_offenders'].fillna(0) + df['juvenile_offenders'].fillna(0)
    
    return df

# Apply decoding
df_ir_decoded = decode_ir(df_ir)

print(" IR DataFrame decoded")
print(f"   - Added offense_1_desc, location_1_desc, bias_1_desc")
print(f"   - Added offender_race_desc, offender_ethnicity_desc")
print(f"   - Calculated total_victims, total_offenders")

 IR DataFrame decoded
   - Added offense_1_desc, location_1_desc, bias_1_desc
   - Added offender_race_desc, offender_ethnicity_desc
   - Calculated total_victims, total_offenders


In [31]:
# ==============================================================================
# DATA QUALITY CHECK
# ==============================================================================

print("\n" + "=" * 80)
print("DATA QUALITY REPORT")
print("=" * 80)

# Check offense codes
print("\n OFFENSE CODES:")
mapped = set(UCR_OFFENSE_CODES.keys())
actual = df_ir['ucr_offense_code_1'].dropna()
unmapped = actual[~actual.isin(mapped)]

if len(unmapped) > 0:
    print(f"  Found {len(unmapped)} unmapped codes:")
    print(unmapped.value_counts().head(10))
else:
    print(" All offense codes mapped!")

# Check bias codes
print("\n BIAS MOTIVATION CODES:")
mapped = set(BIAS_MOTIVATION_CODES.keys())
actual = df_ir['bias_motivation_1'].dropna()
unmapped = actual[~actual.isin(mapped)]

if len(unmapped) > 0:
    print(f"  Found {len(unmapped)} unmapped codes:")
    print(unmapped.value_counts().head(10))
else:
    print(" All bias codes mapped!")

# Check location codes
print("\n LOCATION CODES:")
mapped = set(LOCATION_CODES.keys())
actual = df_ir['location_code_1'].dropna()
unmapped = actual[~actual.isin(mapped)]

if len(unmapped) > 0:
    print(f"  Found {len(unmapped)} unmapped codes:")
    print(unmapped.value_counts().head(10))
else:
    print(" All location codes mapped!")

print("\n" + "=" * 80)


DATA QUALITY REPORT

 OFFENSE CODES:
  Found 223 unmapped codes:
ucr_offense_code_1
23*    85
26F    80
720    14
26G    13
26E    11
36B     5
40B     4
40A     3
510     2
64B     2
Name: count, dtype: int64

 BIAS MOTIVATION CODES:
 All bias codes mapped!

 LOCATION CODES:
  Found 1220 unmapped codes:
location_code_1
58    587
54    109
39     70
48     70
47     62
42     60
37     48
41     45
38     39
45     36
Name: count, dtype: int64



In [32]:
# ==============================================================================
# FINAL TABLE 1: BATCH HEADER (BH) - AGENCY DATABASE
# ==============================================================================

bh_final_columns = {
    'ori': 'agency_id', 'agency_name': 'agency_name', 'city_name': 'city',
    'state_code': 'state_code', 'state_abbr': 'state_abbr', 'state_name': 'state_name',
    'country_region': 'region_code', 'region_name': 'region',
    'country_division': 'division_code', 'division_name': 'division',
    'agency_indicator': 'agency_type_code', 'agency_type': 'agency_type',
    'population_group': 'population_group_code', 'population_group_desc': 'population_group',
    'current_population': 'population', 'last_population': 'population_previous',
    'is_core_city': 'is_core_city', 'is_nibrs_active': 'is_nibrs_reporting',
    'date_ori_added': 'date_agency_added', 'date_ori_went_nibrs': 'date_nibrs_started',
    'inactive_date': 'date_inactive', 'master_file_year': 'reporting_year',
    'ucr_county_code': 'county_code_1', 'msa_code': 'msa_code_1',
    'fips_counties': 'fips_county_codes'
}

df_bh_final = df_bh_decoded.copy()
df_bh_final = df_bh_final[[col for col in bh_final_columns.keys() if col in df_bh_final.columns]]
df_bh_final = df_bh_final.rename(columns=bh_final_columns)

# Add computed columns
df_bh_final['years_since_nibrs'] = (
    pd.to_datetime('2024-12-31') - df_bh_final['date_nibrs_started']
).dt.days / 365.25
df_bh_final['is_active'] = df_bh_final['date_inactive'].isna()

# Sort and reset index
df_bh_final = df_bh_final.sort_values(['state_name', 'city', 'agency_name'])
df_bh_final = df_bh_final.reset_index(drop=True)

print("=" * 80)
print("FINAL TABLE 1: BATCH HEADER (BH) - AGENCY DATABASE")
print("=" * 80)
print(f"Total Agencies: {len(df_bh_final):,}")
print(f"Columns: {len(df_bh_final.columns)}")
df_bh_final.head(10)

FINAL TABLE 1: BATCH HEADER (BH) - AGENCY DATABASE
Total Agencies: 104,544
Columns: 27


,agency_id,agency_name,city,state_code,state_abbr,state_name,region_code,region,division_code,division,agency_type_code,agency_type,population_group_code,population_group,population,population_previous,is_core_city,is_nibrs_reporting,date_agency_added,date_nibrs_started,date_inactive,reporting_year,county_code_1,msa_code_1,fips_county_codes,years_since_nibrs,is_active
0,AL0370100,ABBEVILLE,ABBEVILLE,01,AL,Alabama,3,South,6,East South Central,1,City,6,"Cities 2,500-9,999",2539,0,False,True,NaT,2020-01-01,NaT,2021,034,227,067,4.999316,True
1,AL0370100,ABBEVILLE,ABBEVILLE,01,AL,Alabama,3,South,6,East South Central,1,City,7,"Cities <2,500",2390,0,False,True,NaT,2020-01-01,NaT,2022,034,227,067,4.999316,True
2,AL0370100,ABBEVILLE,ABBEVILLE,01,AL,Alabama,3,South,6,East South Central,1,City,7,"Cities <2,500",2371,0,False,True,NaT,2020-01-01,NaT,2023,034,227,067,4.999316,True
3,AL0370100,ABBEVILLE,ABBEVILLE,01,AL,Alabama,3,South,6,East South Central,1,City,7,"Cities <2,500",2386,0,False,True,NaT,2020-01-01,NaT,2024,034,227,067,4.999316,True
4,AL0370000,HENRY,ABBEVILLE,01,AL,Alabama,3,South,6,East South Central,2,County,9D,"MSA Counties <10,000",9746,0,False,True,NaT,2020-01-01,NaT,2021,034,227,067,4.999316,True
5,AL0370000,HENRY,ABBEVILLE,01,AL,Alabama,3,South,6,East South Central,2,County,9D,"MSA Counties <10,000",9556,0,False,True,NaT,2020-01-01,NaT,2022,034,227,067,4.999316,True
6,AL0370000,HENRY,ABBEVILLE,01,AL,Alabama,3,South,6,East South Central,2,County,9D,"MSA Counties <10,000",9640,0,False,True,NaT,2020-01-01,NaT,2023,034,227,067,4.999316,True
7,AL0370000,HENRY,ABBEVILLE,01,AL,Alabama,3,South,6,East South Central,2,County,9D,"MSA Counties <10,000",9823,0,False,True,NaT,2020-01-01,NaT,2024,034,227,067,4.999316,True
8,AL0012200,ADAMSVILLE,ADAMSVILLE,01,AL,Alabama,3,South,6,East South Central,1,City,6,"Cities 2,500-9,999",4185,0,False,True,NaT,2020-01-01,NaT,2021,037,98,073,4.999316,True
9,AL0012200,ADAMSVILLE,ADAMSVILLE,01,AL,Alabama,3,South,6,East South Central,1,City,6,"Cities 2,500-9,999",4233,0,False,True,NaT,2020-01-01,NaT,2022,037,98,073,4.999316,True


In [33]:
df_bh_final.nunique()

agency_id                26136
agency_name              18658
city                      9024
state_code                  58
state_abbr                  58
state_name                  57
region_code                  5
region                       5
division_code               10
division                    11
agency_type_code             8
agency_type                  8
population_group_code       20
population_group            20
population               27548
population_previous          1
is_core_city                 2
is_nibrs_reporting           1
date_agency_added            0
date_nibrs_started         409
date_inactive                4
reporting_year               4
county_code_1              258
msa_code_1                 427
fips_county_codes         1001
years_since_nibrs          409
is_active                    2
dtype: int64

In [34]:
df_bh_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104544 entries, 0 to 104543
Data columns (total 27 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   agency_id              104544 non-null  object        
 1   agency_name            104544 non-null  object        
 2   city                   89424 non-null   object        
 3   state_code             104544 non-null  object        
 4   state_abbr             104544 non-null  object        
 5   state_name             104544 non-null  object        
 6   region_code            103164 non-null  object        
 7   region                 103164 non-null  object        
 8   division_code          103164 non-null  object        
 9   division               104544 non-null  object        
 10  agency_type_code       101012 non-null  object        
 11  agency_type            101012 non-null  object        
 12  population_group_code  103164 non-null  obje

In [35]:
# ==============================================================================
# FINAL TABLE 2: INCIDENT REPORTS (IR) - HATE CRIME DATABASE
# ==============================================================================

ir_final_columns = {
    'ori': 'agency_id', 'incident_number': 'incident_id',
    'incident_date': 'incident_date', 'bh_index': 'agency_index',
    'state_code': 'state_code', 'state_name': 'state_name',
    'quarter': 'quarter_code', 'quarter_desc': 'quarter',
    'data_source': 'data_source_code', 'data_source_desc': 'data_source',
    'ucr_offense_code_1': 'offense_code', 'offense_1_desc': 'offense_description',
    'location_code_1': 'location_code', 'location_1_desc': 'location',
    'num_victims_1': 'offense_victims',
    'bias_motivation_1': 'bias_code', 'bias_1_desc': 'bias_motivation',
    'bias_1_category': 'bias_category',
    'victim_types_1': 'victim_types_raw', 'victim_types_1_decoded': 'victim_types',
    'num_victims': 'total_victims_reported',
    'adult_victims': 'adult_victims', 'juvenile_victims': 'juvenile_victims',
    'total_victims': 'total_victims',
    'num_offenders': 'total_offenders_reported',
    'adult_offenders': 'adult_offenders', 'juvenile_offenders': 'juvenile_offenders',
    'total_offenders': 'total_offenders',
    'offender_race': 'offender_race_code', 'offender_race_desc': 'offender_race',
    'offender_ethnicity': 'offender_ethnicity_code',
    'offender_ethnicity_desc': 'offender_ethnicity'
}

df_ir_final = df_ir_decoded.copy()
df_ir_final = df_ir_final[[col for col in ir_final_columns.keys() if col in df_ir_final.columns]]
df_ir_final = df_ir_final.rename(columns=ir_final_columns)

# Add time-based columns
df_ir_final['year'] = df_ir_final['incident_date'].dt.year
df_ir_final['month'] = df_ir_final['incident_date'].dt.month
df_ir_final['day_of_week'] = df_ir_final['incident_date'].dt.day_name()
df_ir_final['is_weekend'] = df_ir_final['incident_date'].dt.dayofweek.isin([5, 6])

# Add severity classification
def classify_severity(offense):
    violent = ['Murder', 'Rape', 'Assault', 'Robbery', 'Kidnapping']
    if pd.isna(offense):
        return 'Unknown'
    return 'Violent' if any(v in str(offense) for v in violent) else 'Non-Violent'

df_ir_final['offense_severity'] = df_ir_final['offense_description'].apply(classify_severity)

# Sort and reset index
df_ir_final = df_ir_final.sort_values(['incident_date', 'agency_id', 'incident_id'])
df_ir_final = df_ir_final.reset_index(drop=True)

print("=" * 80)
print("FINAL TABLE 2: INCIDENT REPORTS (IR) - HATE CRIME DATABASE")
print("=" * 80)
print(f"Total Incidents: {len(df_ir_final):,}")
print(f"Columns: {len(df_ir_final.columns)}")
print(f"Total Victims: {df_ir_final['total_victims'].sum():,.0f}")
print(f"Total Offenders: {df_ir_final['total_offenders'].sum():,.0f}")
df_ir_final.head(10)

FINAL TABLE 2: INCIDENT REPORTS (IR) - HATE CRIME DATABASE
Total Incidents: 46,675
Columns: 37
Total Victims: 45,063
Total Offenders: 33,750


,agency_id,incident_id,incident_date,agency_index,state_code,state_name,quarter_code,quarter,data_source_code,data_source,offense_code,offense_description,location_code,location,offense_victims,bias_code,bias_motivation,bias_category,victim_types_raw,victim_types,total_victims_reported,adult_victims,juvenile_victims,total_victims,total_offenders_reported,adult_offenders,juvenile_offenders,total_offenders,offender_race_code,offender_race,offender_ethnicity_code,offender_ethnicity,year,month,day_of_week,is_weekend,offense_severity
0,AL0520100,GM5B8LZG Q8F,2021-01-01,441,01,Alabama,1,Q1 (January - March),N,NIBRS Incident,200,Arson,20,Residence/Home,1,11,Anti-White,Race/Ethnicity,I,Individual,1,1,0,1,1,1,0,1,B,Black or African American,N,Not Hispanic or Latino,2021,1,Friday,False,Non-Violent
1,AZ0080000,V2-IYBRWSU72,2021-01-01,1065,02,Arizona,1,Q1 (January - March),N,NIBRS Incident,120,Robbery,20,Residence/Home,1,28,Anti-Church of Jesus Christ (LDS),Religion,I,Individual,1,1,0,1,1,1,0,1,W,White,N,Not Hispanic or Latino,2021,1,Friday,False,Violent
2,CA0191900,0819PU72ZPX8,2021-01-01,1482,04,California,1,Q1 (January - March),,NaN,13C,Intimidation,20,Residence/Home,1,12,Anti-Black or African American,Race/Ethnicity,I,Individual,2,2,0,2,1,1,0,1,W,White,N,Not Hispanic or Latino,2021,1,Friday,False,Non-Violent
3,CA0331300,CN0BRVSQT28N,2021-01-01,1788,04,California,1,Q1 (January - March),,NaN,13A,Aggravated Assault,13,Highway/Road/Alley,1,12,Anti-Black or African American,Race/Ethnicity,I,Individual,1,1,0,1,1,1,0,1,W,White,N,Not Hispanic or Latino,2021,1,Friday,False,Violent
4,CA0350100,NM-U72841AEM,2021-01-01,1864,04,California,1,Q1 (January - March),,NaN,13C,Intimidation,21,Restaurant,1,41,Anti-Gay (Male),Sexual Orientation,I,Individual,1,1,0,1,1,1,0,1,U,Unknown,H,Hispanic or Latino,2021,1,Friday,False,Non-Violent
5,CA0431000,CN0BRVSD728N,2021-01-01,2082,04,California,1,Q1 (January - March),N,NIBRS Incident,290,Destruction/Damage/Vandalism of Property,50,Park/Playground,2,21,Anti-Jewish,Religion,BG,"Business, Government",0,0,0,0,0,0,0,0,U,Unknown,,NaN,2021,1,Friday,False,Non-Violent
6,CO0070400,CN0BRVSQ628N,2021-01-01,2403,05,Colorado,1,Q1 (January - March),N,NIBRS Incident,13A,Aggravated Assault,24,Specialty Store,2,31,Anti-Arab,Race/Ethnicity,I,Individual,3,3,0,3,1,1,0,1,W,White,N,Not Hispanic or Latino,2021,1,Friday,False,Violent
7,CO0070600,CN0BROPU728N,2021-01-01,2405,05,Colorado,1,Q1 (January - March),N,NIBRS Incident,13C,Intimidation,18,Parking Lot/Garage,1,32,Anti-Hispanic or Latino,Race/Ethnicity,I,Individual,1,1,0,1,1,1,0,1,I,American Indian/Alaska Native,U,Unknown,2021,1,Friday,False,Non-Violent
8,CO0210100,2W2MPU72INKR,2021-01-01,2460,05,Colorado,1,Q1 (January - March),N,NIBRS Incident,13A,Aggravated Assault,13,Highway/Road/Alley,1,12,Anti-Black or African American,Race/Ethnicity,I,Individual,1,1,0,1,1,1,0,1,W,White,H,Hispanic or Latino,2021,1,Friday,False,Violent
9,DCFBIWA01,ST4BAHPU728N,2021-01-01,2891,98,Unknown,1,Q1 (January - March),N,NIBRS Incident,290,Destruction/Damage/Vandalism of Property,04,Church/Synagogue/Temple,1,12,Anti-Black or African American,Race/Ethnicity,R,Religious Organization,0,0,0,0,1,0,0,0,U,Unknown,U,Unknown,2021,1,Friday,False,Non-Violent


In [36]:
# ==============================================================================
# FINAL TABLE 3: MERGED COMPLETE VIEW
# ==============================================================================

df_complete = df_ir_final.merge(
    df_bh_final,
    left_on='agency_index',
    right_index=True,
    how='left',
    suffixes=('', '_agency')
)

# Select key columns for analysis
analysis_columns = [
    'incident_date', 'year', 'month', 'day_of_week', 'quarter',
    'state_name', 'region', 'division', 'city', 'location',
    'agency_name', 'agency_type', 'population_group', 'population',
    'offense_description', 'offense_severity',
    'bias_category', 'bias_motivation',
    'total_victims', 'adult_victims', 'juvenile_victims',
    'total_offenders', 'adult_offenders', 'juvenile_offenders',
    'victim_types', 'offender_race', 'offender_ethnicity'
]

df_complete_view = df_complete[[c for c in analysis_columns if c in df_complete.columns]]

print("=" * 80)
print("FINAL TABLE 3: COMPLETE MERGED VIEW")
print("=" * 80)
print(f"Total Records: {len(df_complete_view):,}")
print(f"Columns: {len(df_complete_view.columns)}")
df_complete_view.head(10)

FINAL TABLE 3: COMPLETE MERGED VIEW
Total Records: 46,675
Columns: 27


,incident_date,year,month,day_of_week,quarter,state_name,region,division,city,location,agency_name,agency_type,population_group,population,offense_description,offense_severity,bias_category,bias_motivation,total_victims,adult_victims,juvenile_victims,total_offenders,adult_offenders,juvenile_offenders,victim_types,offender_race,offender_ethnicity
0,2021-01-01,2021,1,Friday,Q1 (January - March),Alabama,South,East South Central,CROSSVILLE,Residence/Home,CROSSVILLE,City,"Cities <2,500",1810,Arson,Non-Violent,Race/Ethnicity,Anti-White,1,1,0,1,1,0,Individual,Black or African American,Not Hispanic or Latino
1,2021-01-01,2021,1,Friday,Q1 (January - March),Arizona,South,East South Central,LIVINGSTON,Residence/Home,LIVINGSTON,City,"Cities 2,500-9,999",3112,Robbery,Violent,Religion,Anti-Church of Jesus Christ (LDS),1,1,0,1,1,0,Individual,White,Not Hispanic or Latino
2,2021-01-01,2021,1,Friday,Q1 (January - March),California,South,East South Central,RAINBOW CITY,Residence/Home,RAINBOW CITY,City,"Cities 10,000-24,999",10309,Intimidation,Non-Violent,Race/Ethnicity,Anti-Black or African American,2,2,0,1,1,0,Individual,White,Not Hispanic or Latino
3,2021-01-01,2021,1,Friday,Q1 (January - March),California,South,East South Central,UNIONTOWN,Highway/Road/Alley,UNIONTOWN,City,"Cities <2,500",2102,Aggravated Assault,Violent,Race/Ethnicity,Anti-Black or African American,1,1,0,1,1,0,Individual,White,Not Hispanic or Latino
4,2021-01-01,2021,1,Friday,Q1 (January - March),California,South,East South Central,WEST BLOCTON,Restaurant,WEST BLOCTON,City,"Cities <2,500",1222,Intimidation,Non-Violent,Sexual Orientation,Anti-Gay (Male),1,1,0,1,1,0,Individual,Unknown,Hispanic or Latino
5,2021-01-01,2021,1,Friday,Q1 (January - March),California,West,Pacific,BARROW,Park/Playground,NORTH SLOPE BOROUGH,City,"Cities 10,000-24,999",10698,Destruction/Damage/Vandalism of Property,Non-Violent,Religion,Anti-Jewish,0,0,0,0,0,0,"Business, Government",Unknown,NaN
6,2021-01-01,2021,1,Friday,Q1 (January - March),Colorado,West,Mountain,Fort McDowell,Specialty Store,FORT MCDOWELL TRIBAL,Tribal / BIA Agencies,"Cities <2,500",0,Aggravated Assault,Violent,Race/Ethnicity,Anti-Arab,3,3,0,1,1,0,Individual,White,Not Hispanic or Latino
7,2021-01-01,2021,1,Friday,Q1 (January - March),Colorado,West,Mountain,GILA BEND,Parking Lot/Garage,GILA BEND,City,"Cities <2,500",1880,Intimidation,Non-Violent,Race/Ethnicity,Anti-Hispanic or Latino,1,1,0,1,1,0,Individual,American Indian/Alaska Native,Unknown
8,2021-01-01,2021,1,Friday,Q1 (January - March),Colorado,West,Mountain,KINGMAN,Highway/Road/Alley,KINGMAN,City,"Cities 25,000-49,999",32210,Aggravated Assault,Violent,Race/Ethnicity,Anti-Black or African American,1,1,0,1,1,0,Individual,White,Hispanic or Latino
9,2021-01-01,2021,1,Friday,Q1 (January - March),Unknown,West,Mountain,<NA>,Church/Synagogue/Temple,"DEPT OF PUBLIC SAFETY, PHOENIX",State Agency Enforcement Unit,"Cities <2,500",0,Destruction/Damage/Vandalism of Property,Non-Violent,Race/Ethnicity,Anti-Black or African American,0,0,0,0,0,0,Religious Organization,Unknown,Unknown


In [ ]:
# ==============================================================================
# EXPORT FINAL TABLES
# ==============================================================================

print("\n" + "=" * 80)
print("EXPORTING FINAL TABLES")
print("=" * 80)

# Export CSV files
df_bh_final.to_csv('data/FINAL_BH_Agencies.csv', index=False, encoding='utf-8')
print(f"✅ Saved: data/FINAL_BH_Agencies.csv ({len(df_bh_final):,} rows)")

df_ir_final.to_csv('data/FINAL_IR_Incidents.csv', index=False, encoding='utf-8')
print(f"✅ Saved: data/FINAL_IR_Incidents.csv ({len(df_ir_final):,} rows)")

df_complete_view.to_csv('data/FINAL_COMPLETE_HateCrimes.csv', index=False, encoding='utf-8')
print(f" Saved: data/FINAL_COMPLETE_HateCrimes.csv ({len(df_complete_view):,} rows)")

# Export Excel workbook
with pd.ExcelWriter('data/FINAL_FBI_HateCrime.xlsx', engine='openpyxl') as writer:
    df_bh_final.to_excel(writer, sheet_name='Agencies', index=False)
    df_ir_final.to_excel(writer, sheet_name='Incidents', index=False)
    df_complete_view.to_excel(writer, sheet_name='Complete_View', index=False)
    
    # Add summary sheet
    summary = pd.DataFrame({
        'Metric': ['Total Agencies', 'Total Incidents', 'Total Victims', 'Total Offenders'],
        'Value': [
            len(df_bh_final),
            len(df_ir_final),
            int(df_ir_final['total_victims'].sum()),
            int(df_ir_final['total_offenders'].sum())
        ]
    })
    summary.to_excel(writer, sheet_name='Summary', index=False)

print(f" Saved: data/FINAL_FBI_HateCrime.xlsx (4 sheets)")
print("=" * 80)


EXPORTING FINAL TABLES
✅ Saved: data/FINAL_BH_Agencies_2024.csv (104,544 rows)
✅ Saved: data/FINAL_IR_Incidents_2024.csv (46,675 rows)
 Saved: data/FINAL_COMPLETE_HateCrimes_2024.csv (46,675 rows)
 Saved: data/FINAL_FBI_HateCrime_2024.xlsx (4 sheets)


In [38]:
# ==============================================================================
# COMPREHENSIVE DATA SUMMARY
# ==============================================================================

print("\n" + "=" * 80)
print("COMPREHENSIVE DATA SUMMARY - FBI HATE CRIME 2024")
print("=" * 80)

print("\n AGENCIES (BH)")
print("-" * 80)
print(f"Total Agencies: {len(df_bh_final):,}")
print(f"Active Agencies: {df_bh_final['is_active'].sum():,}")
print(f"NIBRS Reporting: {df_bh_final['is_nibrs_reporting'].sum():,}")
print(f"\nTop 5 States:")
print(df_bh_final['state_name'].value_counts().head())

print("\n" + "=" * 80)
print(" INCIDENTS (IR)")
print("-" * 80)
print(f"Total Incidents: {len(df_ir_final):,}")
print(f"Total Victims: {df_ir_final['total_victims'].sum():,.0f}")
print(f"Total Offenders: {df_ir_final['total_offenders'].sum():,.0f}")
print(f"\nTop 5 States:")
print(df_ir_final['state_name'].value_counts().head())
print(f"\nTop 5 Offenses:")
print(df_ir_final['offense_description'].value_counts().head())
print(f"\nBias Categories:")
print(df_ir_final['bias_category'].value_counts())
print(f"\nOffense Severity:")
print(df_ir_final['offense_severity'].value_counts())

print("\n" + "=" * 80)
print(" PIPELINE COMPLETE!")
print("=" * 80)


COMPREHENSIVE DATA SUMMARY - FBI HATE CRIME 2024

 AGENCIES (BH)
--------------------------------------------------------------------------------
Total Agencies: 104,544
Active Agencies: 83,948
NIBRS Reporting: 67,364

Top 5 States:
state_name
Pennsylvania    8592
Texas           7044
Florida         5468
New York        5064
Illinois        4868
Name: count, dtype: int64

 INCIDENTS (IR)
--------------------------------------------------------------------------------
Total Incidents: 46,675
Total Victims: 45,063
Total Offenders: 33,750

Top 5 States:
state_name
California    7856
New Jersey    4384
New York      3665
Washington    2157
Texas         2052
Name: count, dtype: int64

Top 5 Offenses:
offense_description
Intimidation                                13986
Destruction/Damage/Vandalism of Property    11534
Simple Assault                              10288
Aggravated Assault                           5293
All Other Larceny                            1330
Name: count, dtype: in